# Massive production of cancer synthetic RNA-Seq Gene Expression Samples - Tester

This jupyter presents the code for testing the WGAN-GP for producing synthetic samples related to a given cohort.

In [1]:
import time
import torch
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import torch
from torch import cuda,FloatTensor
from torch.nn import Sequential,Linear,LeakyReLU, BatchNorm1d, Module,Tanh ## Clases de redes neuronales
from torch.utils.data import DataLoader,Dataset,random_split
from torch.autograd import Variable, grad
from torch.optim import Adam, RMSprop

from scipy.stats import ks_2samp


import umap.umap_ as umap
from sklearn.preprocessing import StandardScaler


In [6]:
#Checking available devices
print(cuda.device_count())
print(cuda.get_device_name(0))
print(cuda.get_device_properties(0))

1
NVIDIA GeForce MX350
_CudaDeviceProperties(name='NVIDIA GeForce MX350', major=6, minor=1, total_memory=2047MB, multi_processor_count=5)


In [ ]:
class Generator(Module):

    def __init__(self, z_length = 10, output_size: int = None) -> None:
        super(Generator,self).__init__()
        
        self.model = Sequential(
            Linear(z_length,32),
            LeakyReLU(0.2, inplace=True),

            Linear(32,64),
            BatchNorm1d(64,0.8),
            LeakyReLU(0.2, inplace=True),

            Linear(64,output_size),
            BatchNorm1d(output_size,0.8),
            Tanh()
        )

    def forward(self, z):
        return self.model(z)


## Loading data and creating dataloaders

In [ ]:
#Loading expression matrix
cohort = "PRAD" #Options: BRCA, LUAD, PRAD, THCA
exp_matrix = pd.read_csv(f'../data/{cohort}/tumor_exp_matrix.csv',sep=';')

exp_matrix = np.array(exp_matrix.values)
scaler_exp_matrix = StandardScaler().fit(exp_matrix)
scaled_exp_matrix = scaler_exp_matrix.transform(exp_matrix)

In [ ]:
#Preparing data for clipping
min_values = exp_matrix.min(axis=0)
max_values = exp_matrix.max(axis=0)

std_values = np.sqrt(scaler_exp_matrix.var_)

## Initialising model

In [ ]:
# Initialize generator and discriminator
generator = Generator(output_size=exp_matrix.shape[1])

if cuda:
    generator.cuda()

In [ ]:
#Loading generator from model
generator.load_state_dict(torch.load(f'./models/generator_{cohort}.mdl'))

# Generating Samples by the trained network

In [ ]:
def generate_samples(n: int = None):
    generator.eval()
    z = Variable(torch.Tensor(np.random.normal(0, 1, (n, 10)))).cuda()
    fake_samples = generator(z)
    return fake_samples

synthetic_samples = pd.DataFrame(generate_samples(exp_matrix.shape[0]).cpu().data.numpy())

In [ ]:
restituted_synth_matrix = pd.DataFrame(scaler_exp_matrix.inverse_transform(synthetic_samples.values))

restituted_synth_matrix = pd.DataFrame(np.clip(restituted_synth_matrix,
                                               a_min=min_values,
                                               a_max=max_values))
restituted_synth_matrix

## Uniform Maniforld Approximation and Projection (UMAP) results

In [ ]:
reducer = umap.UMAP(random_state=100)

combined_data = np.concatenate((scaled_exp_matrix,synthetic_samples.values))
emb_2d = reducer.fit_transform(combined_data)

cmap = plt.cm.jet

x_1 = [el[0] for el in emb_2d[:scaled_exp_matrix.shape[0]]]
y_1 = [el[1] for el in emb_2d[:scaled_exp_matrix.shape[0]]]

plt.figure(figsize=(10, 10))
plt.scatter(x_1,y_1,alpha=0.5, c = 'gray', s=100, lw = 0, label='Real')


x_2 = [el[0] for el in emb_2d[scaled_exp_matrix.shape[0]:]]
y_2 = [el[1] for el in emb_2d[scaled_exp_matrix.shape[0]:]]
plt.scatter(x_2,y_2, alpha=0.5, c='red', s=50, lw = 0, label='Fake')

plt.axis('off')
plt.savefig(f'../../plots/{cohort}-umap.pdf',dpi = 300, format='pdf', bbox_inches="tight")

In [ ]:
#KL divergence - Two distributions
# Create 2D histograms for the distributions
hist_dist1,x_edges, y_edges = np.histogram2d(x_1, y_1, bins=20, density=True)
hist_dist2, _, _ = np.histogram2d(x_2, y_2, bins=(x_edges, y_edges), density=True)

# Normalize histograms
hist_dist1 /= np.sum(hist_dist1)
hist_dist2 /= np.sum(hist_dist2)

hist_dist1 = hist_dist1.flatten() 
hist_dist2 = hist_dist2.flatten()

In [ ]:
print(ks_2samp(y_1,y_2))
print(ks_2samp(x_1,x_2))

In [ ]:
def calculate_bhatt_distance(pdf1,pdf2):
    # Calculate Bhattacharyya coefficient
    bc = np.sum(np.sqrt(pdf1 * pdf2))

    # Calculate Bhattacharyya distance
    bd = -np.log(bc)
    return bd

In [ ]:
# Calculate Bhattacharyya distance
bhatt_dist = calculate_bhatt_distance(hist_dist1,hist_dist2)
print(f"Bhattacharyya Distance = {bhatt_dist}")

## Principal component analysis (PCA) Results

In [ ]:
from sklearn.decomposition import PCA

reducer = PCA(n_components=2, random_state=0)

combined_data = np.concatenate((scaled_exp_matrix,synthetic_samples.values))
emb_2d = reducer.fit_transform(combined_data)

cmap = plt.cm.jet

x_1 = [el[0] for el in emb_2d[:scaled_exp_matrix.shape[0]]]
y_1 = [el[1] for el in emb_2d[:scaled_exp_matrix.shape[0]]]

plt.figure(figsize=(10, 10))
plt.scatter(x_1,y_1,alpha=0.5, c='gray', s=100, lw = 0, label='Real')

x_2 = [el[0] for el in emb_2d[scaled_exp_matrix.shape[0]:]]
y_2 = [el[1] for el in emb_2d[scaled_exp_matrix.shape[0]:]]
plt.scatter(x_2,y_2, alpha=0.5, c='red', s=50, lw = 0, label='Fake')

plt.axis('off')
plt.savefig(f'../../plots/{cohort}-pca.pdf',dpi = 300, format='pdf', bbox_inches="tight")

In [ ]:
#KL divergence - Two distributions
# Create 2D histograms for the distributions
hist_dist1,x_edges, y_edges = np.histogram2d(x_1, y_1, bins=20, density=True)
hist_dist2, _, _ = np.histogram2d(x_2, y_2, bins=(x_edges, y_edges), density=True)

# Normalize histograms
hist_dist1 /= np.sum(hist_dist1)
hist_dist2 /= np.sum(hist_dist2)

hist_dist1 = hist_dist1.flatten() 
hist_dist2 = hist_dist2.flatten()

In [ ]:
print(ks_2samp(y_1,y_2))
print(ks_2samp(x_1,x_2))

In [ ]:
def calculate_bhatt_distance(pdf1,pdf2):
    # Calculate Bhattacharyya coefficient
    bc = np.sum(np.sqrt(pdf1 * pdf2))

    # Calculate Bhattacharyya distance
    bd = -np.log(bc)
    return bd

In [ ]:
# Calculate Bhattacharyya distance
bhatt_dist = calculate_bhatt_distance(hist_dist1,hist_dist2)
print(f"Bhattacharyya Distance = {bhatt_dist}")

## Quantitative analysis of the results

From this cell, it starts the quantitative analysis of the results, which includes the observation of the relative error of the average expression value for each of the genes. Furthermore, the correlation between synthetic and real data is analysed too.

In [ ]:
real_matrix = pd.DataFrame(exp_matrix)
synth_matrix = pd.DataFrame(restituted_synth_matrix)

### Relative error for each gene

In [ ]:
means_real_matrix = real_matrix[real_matrix.columns].mean().values
means_synth_matrix = synth_matrix[synth_matrix.columns].mean().values

relative_error = abs(means_real_matrix-means_synth_matrix)/means_real_matrix

np.mean(relative_error)

### Correlation between values

In [ ]:
def upper_diag_list(m_):
    """
    Returns the condensed list of all the values in the upper-diagonal of m_
    :param m_: numpy array of float. Shape=(N, N)
    :return: list of values in the upper-diagonal of m_ (from top to bottom and from
             left to right). Shape=(N*(N-1)/2,)
    """
    m = np.triu(m_, k=1)  # upper-diagonal matrix
    tril = np.zeros_like(m_) + np.nan
    tril = np.tril(tril)
    m += tril
    m = np.ravel(m)
    return m[~np.isnan(m)]


In [ ]:
def gamma_coef(x, y):
    """
    Compute gamma coefficients for two given expression matrices
    :param x: matrix of gene expressions. Shape=(nb_samples_1, nb_genes)
    :param y: matrix of gene expressions. Shape=(nb_samples_2, nb_genes)
    :return: Gamma(D^X, D^Z)
    """
    dists_x = pd.DataFrame(1 - upper_diag_list(x.corr()))
    dists_y = pd.DataFrame(1 - upper_diag_list(y.corr()))
    gamma_dx_dy = dists_x.corrwith(dists_y, axis=0)
    return gamma_dx_dy


In [ ]:
gamma_coef(pd.DataFrame(exp_matrix),restituted_synth_matrix)